# 💬 Sesiunea 14–15 · NLP: de la text la predicții

**Povestea de azi:** lucrezi ca data scientist la un magazin online. Ai **~28.000 de recenzii reale în limba română** (produse de pe eMAG și filme) și vrei să răspunzi la două întrebări:

| Întrebare | Tehnica NLP |
|---|---|
| Recenzia e **pozitivă sau negativă**? | Clasificare de text (sentiment) |
| **Ce nume de persoane și locuri** apar în mesaje? | Named Entity Recognition (NER) |

Drumul: **text brut → curățare → numere → model → interpretare**.

## Pregătire (rulează o dată)

In [ ]:
# modelul spaCy pentru limba română (pentru lematizare și NER)
!python -m spacy download ro_core_news_sm -q

import re, unicodedata, importlib, warnings
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

import spacy
from spacy.lang.ro.stop_words import STOP_WORDS
plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_colwidth", 120)

URL = "https://raw.githubusercontent.com/dumitrescustefan/Romanian-Transformers/examples/examples/sentiment_analysis/ro/"
train = pd.read_csv(URL + "train.csv").dropna(subset=["text"])[["text", "label"]]
test  = pd.read_csv(URL + "test.csv").dropna(subset=["text"])[["text", "label"]]
print("gata:", len(train), "recenzii train /", len(test), "recenzii test")

## 1 · Textul ca date 👀

Înainte de orice model, ne uităm la date. `label = 1` înseamnă recenzie **pozitivă**, `label = 0` **negativă**.

In [ ]:
train.sample(8, random_state=7)

In [ ]:
print("Distribuția etichetelor (train):")
print(train["label"].value_counts().rename({1: "pozitiv", 0: "negativ"}))

train["nr_cuvinte"] = train["text"].str.split().str.len()
train["nr_cuvinte"].clip(upper=300).hist(bins=40)
plt.title("Câte cuvinte are o recenzie?"); plt.xlabel("nr. cuvinte (tăiat la 300)"); plt.show()

**🤔 Întrebare** · Rulează de câteva ori `train.sample(8)` (fără `random_state`). Ce „murdărie” vezi în text, care ne-ar putea încurca?

> ✍️ *Răspunsul tău:* ....

## 2 · Preprocesare: curățăm textul 🧹
*(după slide-ul „Pipeline-ul NLP”)*

Modelul vede doar șiruri de caractere. Pentru el, `Bun`, `bun`, `bun!` și `bún` sunt 4 cuvinte diferite. Curățarea le aduce la aceeași formă.

**Exercițiul 2.1 · Funcția de curățare**

Completează cele două linii marcate cu `# ...` din funcția de mai jos. Apoi testează pe propoziția ta (scrie cu diacritice, majuscule, semne!).

In [ ]:
def fara_diacritice(text):
    # "ăâîșț" -> "aaist": separăm litera de semnul diacritic și păstrăm doar litera
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")

def curata_text(text):
    text = # ...                                 # 1. litere mici
    text = fara_diacritice(text)                 # 2. fără diacritice (ca în setul de date)
    text = re.sub(r"<br\s*/?>|\bbr\b", " ", text)  # 3. resturi de HTML
    text = # ...                                 # 4. păstrăm doar litere și spații (indiciu: re.sub cu [^a-z\s])
    text = re.sub(r"\s+", " ", text).strip()     # 5. spații multiple -> unul singur
    return text

exemplu = "Telefonul e SUPER!!! Livrat în 2 zile, dar încărcătorul... nu prea merge :("
print(curata_text(exemplu))

In [ ]:
# ✏️ încearcă și tu: scrie orice propoziție
print(curata_text("Scrie aici o recenzie a ta, cu DIACRITICE, cifre 123 și emoji 😀!"))

**🤔 Întrebare** · Am șters toate cifrele. Când ar fi o idee proastă?

> ✍️ *Răspunsul tău:* ....

### Stopwords: cuvinte foarte dese, cu (aparent) puțină informație
„și”, „de”, „la”, „un”... apar peste tot. De obicei le scoatem. Dar atenție ce e în listă!

**Exercițiul 2.2 · Capcana stopwords**

Rulează celula. Apoi adaugă propriile perechi de propoziții în listă. Ce se întâmplă cu sensul?

> ✍️ *Răspunsul tău:* ....

In [ ]:
STOP = {fara_diacritice(w) for w in STOP_WORDS}
print("câte stopwords:", len(STOP))
print("conține 'nu'?", "nu" in STOP, "| 'foarte'?", "foarte" in STOP, "| 'bine'?", "bine" in STOP)

def fara_stopwords(text):
    return " ".join(w for w in curata_text(text).split() if w not in STOP)

for propozitie in ["Nu e bun.", "E bun.", "Nu recomand acest telefon.", "Recomand acest telefon."]:   # ✏️ adaugă și tu
    print(f"{propozitie:32s} ->  '{fara_stopwords(propozitie)}'")

### Lematizare: aducem cuvintele la forma de dicționar
`coletul`, `coletele`, `coletului` → `colet`. Așa modelul învață un singur cuvânt în loc de cinci. Folosim **spaCy**, care are un model antrenat pentru română.

In [ ]:
importlib.invalidate_caches()
nlp = spacy.load("ro_core_news_sm")

doc = nlp("Coletele au ajuns repede, dar telefoanele erau zgâriate.")
for token in doc:
    print(f"{token.text:12s} -> {token.lemma_}")

**🤔 Întrebare** · Uită-te atent la lemele de mai sus. Sunt toate corecte?

> ✍️ *Răspunsul tău:* ....

## 3 · Vectorizare: din cuvinte în numere 🔢
*(după slide-urile „Bag of Words” și „TF-IDF”)*

Modelele ML lucrează cu numere. Pornim de la 4 recenzii mini, ca să vedem matricea cu ochiul liber.

**Exercițiul 3.1 · Bag of Words vs TF-IDF**

Completează linia cu `CountVectorizer`, rulează, apoi compară cele două tabele.

> ✍️ *Răspunsul tău:* ....

In [ ]:
mini = ["telefon bun si ieftin",
        "telefon prost",
        "nu e bun",
        "livrare rapida telefon bun"]

bow = CountVectorizer()
X_bow = bow.fit_transform(mini)
print("Bag of Words (numărăm cuvintele):")
display(pd.DataFrame(X_bow.toarray(), columns=bow.get_feature_names_out(), index=mini))

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(mini)
print("TF-IDF (cuvintele rare cântăresc mai mult):")
display(pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out(), index=mini).round(2))

**N-grame:** în loc de cuvinte izolate, luăm și perechi de cuvinte vecine. Așa „nu e” sau „e bun” devin coloane separate.

In [ ]:
bigrame = CountVectorizer(ngram_range=(1, 2)) # folosește și unigrame și bigrame
bigrame.fit(mini)
print(list(bigrame.get_feature_names_out()))

Observă: **„e” a dispărut** din „nu e bun”. Implicit, `CountVectorizer` ignoră cuvintele de o singură literă. Detaliile de felul ăsta contează în română („e”, „o”, „a”).

**🤔 Întrebare** · Ai 28.000 de recenzii și un vocabular de ~50.000 de cuvinte. Cum arată matricea Bag of Words? Ce e special la ea?

> ✍️ *Răspunsul tău:* ....

## 4 · Clasificare de sentiment 🙂🙁
*(după slide-ul „Clasificare de text”)*

Rețeta e cea pe care o știi deja din sesiunile de clasificare: **features → model → evaluare pe test**. Singura noutate: features-urile vin din TF-IDF.

**Exercițiul 4.1 · Primul clasificator de sentiment**

Completează pipeline-ul: un `TfidfVectorizer` (cu `preprocessor=curata_text` și `min_df=3`) urmat de o `LogisticRegression(max_iter=1000)`.

In [ ]:
model = make_pipeline(
    # ...   (TfidfVectorizer)
    # ...   (LogisticRegression)
)
model.fit(train["text"], train["label"])

pred = model.predict(test["text"])
print(f"Acuratețe pe test: {accuracy_score(test['label'], pred):.3f}")
print(f"Dacă am ghici mereu 'pozitiv': {(test['label'] == 1).mean():.3f}\n")
print(classification_report(test["label"], pred, target_names=["negativ", "pozitiv"]))
ConfusionMatrixDisplay.from_predictions(test["label"], pred, display_labels=["negativ", "pozitiv"], cmap="Blues")
plt.xlabel("ce spune modelul"); plt.ylabel("realitate"); plt.title("Matricea de confuzie"); plt.show()

### Ce a învățat modelul?
Regresia logistică dă fiecărui cuvânt un **coeficient**: pozitiv împinge spre „pozitiv”, negativ spre „negativ”.

In [ ]:
vocab = model[0].get_feature_names_out()
coef = model[-1].coef_[0]
top = pd.Series(coef, index=vocab).sort_values()
cuvinte = pd.concat([top.head(12), top.tail(12)])

cuvinte.plot.barh(color=["#d9534f"] * 12 + ["#5cb85c"] * 12, figsize=(7, 7))
plt.title("Cuvintele cu cea mai mare influență"); plt.axvline(0, color="k", lw=0.8); plt.show()

**Exercițiul 4.2 · Păcălește modelul 😈**

Scrie în listă recenziile tale (fără griji pentru diacritice, le curăță `curata_text`). Poți găsi o recenzie **clar pozitivă** pe care modelul o vede negativă? Sau invers?

> ✍️ *Răspunsul tău:* ....

In [ ]:
recenziile_mele = [
    "Produsul este excelent, recomand cu drag!",
    "Nu e rău deloc.",
    "Super, s-a stricat a doua zi.",
    "Livrare rapidă, dar produsul e prost.",
    # ✏️ adaugă aici recenziile tale
]
proba = model.predict_proba(recenziile_mele)[:, 1]
for text, p in zip(recenziile_mele, proba):
    eticheta = "🙂 pozitiv" if p >= 0.5 else "🙁 negativ"
    print(f"{p:.2f}  {eticheta:11s}  {text}")

### Unde greșește pe test?

In [ ]:
greseli = test.assign(pred=pred, p_pozitiv=model.predict_proba(test["text"])[:, 1])
greseli = greseli[greseli["label"] != greseli["pred"]]
greseli[greseli["text"].str.len() < 200].sample(6, random_state=1)[["text", "label", "pred", "p_pozitiv"]]

**🤔 Întrebare** · Uită-te la greșelile de mai sus. Sunt toate „vina” modelului?

> ✍️ *Răspunsul tău:* ....

**Exercițiul 4.3 · Poți face un model mai bun?**

Adaugă în dicționar 1–2 variante proprii și rulează. Idei: `ngram_range=(1, 2)`, `MultinomialNB()`, `min_df=10`, `max_features=5000`.
Notează: ce variantă câștigă și cu cât?

> ✍️ *Răspunsul tău:* ....

In [ ]:
variante = {
    "TF-IDF + LogReg (baseline)": make_pipeline(TfidfVectorizer(preprocessor=curata_text, min_df=3),
                                               LogisticRegression(max_iter=1000)),
    # ✏️ "numele variantei tale": make_pipeline(...),
    # ✏️ "încă o variantă":       make_pipeline(...),
}
for nume, m in variante.items():
    m.fit(train["text"], train["label"])
    print(f"{nume:32s} acuratețe test = {m.score(test['text'], test['label']):.3f}")

## 5 · Named Entity Recognition (NER) 🏷️

NER identifică în text entități precum persoane, organizații, locuri sau date. Cu spaCy, folosim un model pre-antrenat:

1. Îl încărcăm o singură dată: `nlp = spacy.load("ro_core_news_sm")`.
2. Îi trimitem textul: `doc = nlp(text)`.
3. Citim entitățile găsite: `doc.ents`. Pentru fiecare, `ent.text` este textul identificat, iar `ent.label_` este categoria.

**Variante utile:** `displacy.render(doc, style="ent", jupyter=True)` evidențiază entitățile vizual; `nlp.pipe(lista_de_texte)` procesează mai multe texte; `EntityRuler` permite reguli simple pentru nume cunoscute. Pentru alte limbi se încarcă un model potrivit. Un model poate rata sau clasifica greșit entități, mai ales nume de firme specifice.

In [18]:
import spacy
from spacy import displacy

nlp = spacy.load("ro_core_news_sm")
text = "Ana Ionescu locuiește în București."
doc = nlp(text)

for entitate in doc.ents:
    print(entitate.text, entitate.label_)

displacy.render(doc, style="ent", jupyter=True)

Ana Ionescu PERSON
București GPE


**Exercițiu · Încearcă un exemplu**

Schimbă propoziția din celula de mai sus și observă ce entități identifică modelul.

**Temă · Adaugă reguli personalizate la NER**

Modelul spaCy nu recunoaște întotdeauna numele firmelor pe care le întâlnim local. Folosește `EntityRuler` ca să adaugi o regulă care etichetează `eMAG` și `Dedeman` cu eticheta `COMPANIE`. Apoi adaugă încă o firmă și testează toate numele într-o propoziție proprie.

Scrie codul în celula următoare, folosind modelul românesc `ro_core_news_sm`. La final, explică în cuvintele tale:

1. Ce face `EntityRuler` și cu ce diferă de modelul NER pre-antrenat?
2. Ce reprezintă `label` și `pattern` în fiecare regulă?
3. De ce adăugăm ruler-ul cu `before="ner"`?
4. Ce se întâmplă dacă apare în text o firmă pentru care nu am scris o regulă?

In [ ]:
# Scrie aici soluția ta.
# Indicii: nlp.add_pipe("entity_ruler", before="ner") și ruler.add_patterns(...)
# Procesează apoi o propoziție și afișează textul și eticheta fiecărei entități.

## 6 · Recap 🎓

**Text brut → curățare → numere → model → interpretare**

- **Preprocesare:** lowercase, diacritice, punctuație, stopwords, lematizare. Alegem pașii după sarcină (pentru sentiment păstrăm „nu”, pentru NER păstrăm majusculele).
- **Vectorizare:** Bag of Words numără, TF-IDF cântărește, n-gramele păstrează puțin din ordine.
- **Clasificare de sentiment:** TF-IDF + regresie logistică, ~86% pe recenzii reale. Limite: negații, sarcasm, recenzii mixte.
- **NER (spaCy):** identifică nume și locuri într-un text; păstrăm majusculele.